In [0]:
# primero creamos el volumen y subimos los archicos 
catalog = "workspace"  
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.electiva")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.electiva.datos_csv")

In [0]:
# verifico que quedaron los 8 archivos csv
display(dbutils.fs.ls("/Volumes/workspace/electiva/datos_csv"))

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"     
schema  = "electiva"
volume  = "datos_csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

vol_path = f"dbfs:/Volumes/{catalog}/{schema}/{volume}"

archivos = {
    "customers.csv":            "customers",
    "employees.csv":            "employee",
    "orders.csv":               "orders",
    "productcategories.csv":    "product_categories",
    "productsubcategories.csv": "product_sub_categories",
    "products.csv":             "products",
    "vendors.csv":              "vendor",
    "vendorproduct.csv":        "vendor_product",
}

for archivo, tabla in archivos.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("encoding", "ISO-8859-1")
          .option("nullValue", "NULL")
          .csv(f"{vol_path}/{archivo}"))

    if tabla == "orders":
        for c in ["OrderDate", "DueDate", "ShipDate"]:
            df = df.withColumn(c, F.coalesce(
                F.try_to_timestamp(F.col(c).cast("string"), F.lit("M/d/yyyy")),
                F.try_to_timestamp(F.col(c).cast("string"))
            ).cast("date"))

    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{catalog}.{schema}.{tabla}"))
    print(f"{tabla}: {df.count()} filas")

1. primer punto construcción del campo user_name mediante la concatenación de la información correspondiente a cada cliente.

In [0]:
%sql
SELECT
  customerid,
  fullname,
  concat(customerid, '-',
         upper(substring(firstname, 1, 2)), '-',
         lower(substring(lastname, 1, 2)), '-AW') AS user_name
FROM customers
ORDER BY lastname, firstname;
     

2. Segundo punto diseño y creación de la tabla page_user para almacenar la información solicitada.

In [0]:
%sql
CREATE OR REPLACE TABLE page_user (customerid INT, user_name  STRING) USING DELTA;

se ponen los datos

In [0]:
%sql
INSERT INTO page_user
SELECT customerid, concat(customerid, '-', upper(substring(firstname, 1, 2)), '-', lower(substring(lastname, 1, 2)), '-AW') AS user_name
FROM customers;

In [0]:
%sql
SELECT * FROM page_user ORDER BY customerid;

3. Tercer punto generación de una nueva tabla con la información de todas las órdenes registradas durante agosto de 2011.

In [0]:
%sql
CREATE OR REPLACE VIEW v_utl_ordenes_mitad_anio AS
WITH base AS (
  SELECT
    p.productname,
    p.modelname,
    u.user_name,
    o.salesorderid,
    c.categoryname,
    o.orderdate,
    o.linetotal
  FROM orders o
  JOIN products p               ON o.productid     = p.productid
  JOIN product_sub_categories s ON p.subcategoryid = s.subcategoryid
  JOIN product_categories c     ON s.categoryid    = c.categoryid
  JOIN page_user u              ON o.customerid    = u.customerid
  WHERE o.orderdate >= DATE'2011-08-01'
    AND o.orderdate <  DATE'2011-09-01'
),
con_descuento AS (
  SELECT *,
    CASE
      WHEN categoryname ILIKE 'Bikes'       THEN 0.10
      WHEN categoryname ILIKE 'Clothing'    THEN 0.15
      WHEN categoryname ILIKE 'Components'  THEN 0.08
      WHEN categoryname ILIKE 'Accessories' THEN 0.20
      ELSE 0
    END AS `descuento_mitad_año`
  FROM base
)
SELECT *,
       linetotal - linetotal * `descuento_mitad_año` AS total_descuentos_aplicados
FROM con_descuento;
     

In [0]:
%sql
SELECT * FROM v_utl_ordenes_mitad_anio
ORDER BY user_name, salesorderid;

In [0]:
%sql
CREATE OR REPLACE TABLE ordenes_mitad_anio AS
SELECT * FROM v_utl_ordenes_mitad_anio;

4. Cuarto punto implementación de una vista materializada llamada customer_class, destinada a clasificar a los clientes correspondientes al año 2011 según su nivel de actividad. Esta deberá incluir el nombre de usuario (user_name), la categoría asignada (user_level: Oro, Plata u Ocasional), el identificador del cliente (customerid), su nombre completo y la cantidad de órdenes realizadas durante el año

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW customer_class AS
SELECT
  COUNT(DISTINCT o.salesorderid) AS ammount_of_orders,
  u.user_name,
  c.fullname                     AS customerfullname,
  CASE
    WHEN COUNT(DISTINCT o.salesorderid) >= 3 THEN 'Oro'
    WHEN COUNT(DISTINCT o.salesorderid) = 2  THEN 'Plata'
    ELSE 'Ocasional'
  END                            AS user_level,
  c.customerid
FROM orders o
JOIN customers c ON o.customerid = c.customerid
JOIN page_user u ON o.customerid = u.customerid
WHERE year(o.orderdate) = 2011
GROUP BY c.customerid, c.fullname, u.user_name;

In [0]:
%sql
SELECT * FROM customer_class;

In [0]:
%sql
SELECT user_level, COUNT(*) AS clientes
FROM customer_class
GROUP BY user_level;

Conclusión

En el desarrollo de este parcial se realizó la carga y el procesamiento de los datos en Databricks, dando solución a los cuatro puntos planteados: la identificación del user_name de los clientes, la creación de la tabla page_user, la elaboración de la vista correspondiente a los descuentos de agosto y la creación de la vista materializada customer_class. Este trabajo permitió aplicar los conocimientos adquiridos durante el curso y fortalecer el manejo de herramientas y funciones como los joins, las estructuras condicionales CASE WHEN y las vistas. Finalmente, se precisa que para el punto 3 se utilizó la fórmula establecida en el enunciado, mientras que en el punto 4 se presentan los 205 registros obtenidos, sin aplicar el límite de 100 filas que aparecía en el ejemplo de referencia.